In [3]:
# preprocess import numpy as np
from scipy.interpolate import griddata
from datetime import timedelta, datetime, date
import dateutil.parser
import pandas as pd
from scipy.spatial import distance_matrix
import xarray as xr
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [2]:
## 
country = 'cuba' ## FILL WITH THE COUNTRY NAME e.g. cuba

In [4]:
##
data_path = '' ## FILL WITH THE PATH of station data
stations_path  = f'{data_path}allstat_10m_wind.csv' 

downscaled_path = f'../data/downscaled_data/{country}/' ## FILL WITH THE PATH of ERA5LAND data
idw_path = f'../data/idw_data/{country}/' ## FILL WITH THE PATH of ERA5LAND data

output_path = f'../data/atlas_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

In [5]:
month = 1

In [6]:
def opening_datasets(path_era5, path_stats, n_month, country):
    era5_ds = xr.open_dataset(downscaled_path+f'10m_wind_downscaled_{country}_m{n_month}.nc').sortby('latitude')
    #Latitudes have to be in descending order
    era5_ds = era5_ds.reindex(latitude = era5_ds.latitude[::-1])

    idw_ds  = xr.open_dataset(idw_path+f'10m_wind_idw_{country}_m{n_month}.nc')
    
    return era5_ds, idw_ds

In [9]:
era5_ds, idw_ds = opening_datasets(downscaled_path,idw_path,month,country)

$$
\Huge
wind_\mathrm{integrated} = \frac{\frac{\mathrm{wind}_{\mathrm{era5}}}{\sigma_{\mathrm{era5}}^2} + \frac{\mathrm{wind}_{\mathrm{idw}}}{\sigma_{\mathrm{idw}}^2}}{\frac{1}{\sigma_{\mathrm{era5}}^2} + \frac{1}{\sigma_{\mathrm{idw}}^2}}
$$

In [10]:
def apply_dist(pdf, lat_new, lon_new):

    lat = pdf['latitude'].drop_duplicates().values
    lon = pdf['longitude'].drop_duplicates().values

    # Locations of the IoT observations
    iot_grid = np.array(list(zip(lat, lon)))

    # Grid for IoT observation projection
    grid_new = np.array([[x, y] for x in lat_new for y in lon_new])

    # Calculating IoT map
    iot_new = distances(iot_grid, grid_new)

    # Reshaping results from array-like to matrix-like
    iot_reshape = []
    for i in np.arange(0,len(iot_new)):
        iot_reshape.append(iot_new[i].reshape(len(lat_new), len(lon_new)))

    return iot_reshape

In [11]:
def distances(pos, posi):

    # Distance matrix
    dist = distance_matrix(pos, posi)**3         #this is what we need

    # Weight matrix
    #weights = 1.0 / dist

    # Change diagonal values to 1
    #np.fill_diagonal(weights, 1)

    # Normalizing each column of the weight matrix
    #weights = weights/np.nansum(weights, axis=0)

    # Calculating the results of the IDW method
    #zi = np.dot(weights.T, z)

    return dist

In [12]:
def min_distances(idw_ds, stations_path, country):
    lat_fin = idw_ds.latitude.to_numpy()
    lon_fin = idw_ds.longitude.to_numpy()
    stats_data = pd.read_csv(stations_path)
    dist_matrices = apply_dist(stats_data, lat_fin, lon_fin)
    
    for i in np.arange(1,len(dist_matrices)):
        if i == 1:
            min_init = np.minimum(dist_matrices[i],dist_matrices[i-1])
        else:
            min_dist = np.minimum(min_init,dist_matrices[i])
        
    min_dist_xr  = xr.DataArray(min_dist, coords=[("latitude", lat_fin), ("longitude", lon_fin)])

    # NORMALIZATION 
    min_dist_xr_norm = (min_dist_xr - min_dist_xr.min()) / (min_dist_xr.max() - min_dist_xr.min())
    return min_dist_xr_norm

In [13]:
min_dist_xr_norm = min_distances(idw_ds,stations_path,country)

In [14]:
def uncertainties(min_dist_xr_norm):
    #uncertainties of the era5 dataset
    sigma_2_era5_norm= (0.2)**2
    
    #uncertainties of the idw dataset
    sigma_2_idw_norm = (0.4 + min_dist_xr_norm)**2
    
    return sigma_2_era5_norm, sigma_2_idw_norm

In [15]:
sigma_2_era5_norm, sigma_2_idw_norm = uncertainties(min_dist_xr_norm)

In [16]:
def apply_scaling(era5_variable, sigma_2_era5_norm, idw_variable, sigma_2_idw_norm, 
                  n_month, country):
    var_scaled = ((era5_variable / sigma_2_era5_norm) +
                  (idw_variable / sigma_2_idw_norm)) / ((1/sigma_2_era5_norm) + (1/sigma_2_idw_norm))
    
    return var_scaled

In [17]:
era5_ds

<xarray.Dataset>
Dimensions:         (latitude: 3305, longitude: 8889)
Coordinates:
  * latitude        (latitude) float32 23.19 23.19 23.18 ... 19.86 19.86 19.86
  * longitude       (longitude) float32 -84.89 -84.89 -84.88 ... -74.14 -74.14
Data variables:
    u10_downscaled  (latitude, longitude) float64 ...
    v10_downscaled  (latitude, longitude) float64 ...
    wds_downscaled  (latitude, longitude) float64 ...
    dir_downscaled  (latitude, longitude) float64 ...

In [18]:
u10_reshaped = apply_scaling(era5_ds.u10_downscaled, sigma_2_era5_norm,
                            idw_ds.u10_reshaped, sigma_2_idw_norm,
                            month, country)

In [19]:
v10_reshaped = apply_scaling(era5_ds.v10_downscaled, sigma_2_era5_norm,
                            idw_ds.v10_reshaped, sigma_2_idw_norm,
                            month, country)

In [20]:
wds_reshaped = apply_scaling(era5_ds.wds_downscaled, sigma_2_era5_norm,
                            idw_ds.wds_reshaped, sigma_2_idw_norm,
                            month, country)

In [21]:
dir_reshaped = apply_scaling(era5_ds.dir_downscaled, sigma_2_era5_norm,
                            idw_ds.dir_reshaped, sigma_2_idw_norm,
                            month, country)

In [22]:
def creating_geotiff(file_netcdf,country,n_month):
    bT = file_netcdf#['ssr_scaled']
    bT = bT.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
    bT.rio.crs
    # Define the CRS projection
    bT = bT.rio.write_crs("epsg:4326", inplace=True)
    bT.rio.to_raster(output_path+f'10m_wind_integrated_{country}_m{n_month}.tif')
    return bT

In [23]:
def create_scaled_dataset(u_scaled,v_scaled,wds_scaled,dir_scaled):
    u_scaled_xr = u_scaled.to_dataset(name='u10_integrated')
    v_scaled_xr = v_scaled.to_dataset(name='v10_integrated')
    wds_scaled_xr = wds_scaled.to_dataset(name='wds_integrated')
    dir_scaled_xr = dir_scaled.to_dataset(name='dir_integrated')
    wind_sc_month = xr.merge([u_scaled_xr, v_scaled_xr, wds_scaled_xr, dir_scaled_xr]
                                ).assign_attrs(units= 'm s-1, [°]', 
                                               description = 'Wind Speed measurement height: 10m. Wind Direction measurement: degrees North in clockwise turn')
    wind_sc_month.to_netcdf(output_path+f'10m_wind_integrated_{country}_m{month}.nc')
    geotiff_file = creating_geotiff(wind_sc_month,country,month)
    return wind_sc_month

In [24]:
wind_integrated_xr = create_scaled_dataset(u10_reshaped,v10_reshaped, wds_reshaped, dir_reshaped)    